In [6]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# --- CONFIGURACIÓN DE RUTAS (SOLO PARQUET) ---
PROCESSED_FOREX_PATH = "/home/quant/data/processed/forex"
PROCESSED_INDICES_PATH = "/home/quant/data/processed/indices"

# --- CONSTANTES DXY (Pesos Oficiales) ---
DXY_WEIGHTS = {
    'EURUSD': -0.576,
    'USDJPY': 0.136,
    'GBPUSD': -0.119,
    'USDCAD': 0.091,
    'USDSEK': 0.042,
    'USDCHF': 0.036
}
DXY_CONSTANT = 50.14348112

def load_gold_asset(asset_name="xauusd"):
    """
    Carga exclusivamente desde la fuente Parquet procesada.
    """
    print(f"--- INICIANDO CAPA GOLD (CARGA DE ACTIVOS) ---")
    
    asset_name = asset_name.lower()
    parquet_path = os.path.join(PROCESSED_FOREX_PATH, f"{asset_name}_clean.parquet")
    
    if not os.path.exists(parquet_path):
        print(f"❌ Error Crítico: No se encontró la fuente Parquet en {parquet_path}")
        print(f"   Asegúrate de ejecutar primero el proceso de limpieza (Silver Layer).")
        return None

    print(f"📦 Fuente: Parquet. Cargando {asset_name.upper()}...")
    df = pd.read_parquet(parquet_path)
    print(f"  ✅ Cargado: {parquet_path} ({len(df):,} registros)")
    
    return {asset_name.upper(): df}

def calculate_dxy_from_parquet():
    """
    Calcula el DXY si los componentes están disponibles en la carpeta de procesados.
    """
    print(f"\n--- VERIFICANDO COMPONENTES PARA CÁLCULO DXY ---")
    
    dxy_components = {}
    missing = []
    
    for asset in DXY_WEIGHTS.keys():
        path = os.path.join(PROCESSED_FOREX_PATH, f"{asset.lower()}_clean.parquet")
        if os.path.exists(path):
            dxy_components[asset] = pd.read_parquet(path)['Close']
        else:
            missing.append(asset)
            
    if missing:
        print(f"ℹ️ DXY no se puede calcular: faltan Parquets de {', '.join(missing)}")
        return None

    print(f"🔨 Calculando DXY desde componentes Parquet...")
    df_dxy = pd.DataFrame(dxy_components).ffill(limit=5).dropna()
    
    log_dxy = np.log(DXY_CONSTANT)
    for asset, weight in DXY_WEIGHTS.items():
        log_dxy += weight * np.log(df_dxy[asset])
    
    df_dxy['Close'] = np.exp(log_dxy)
    df_dxy['timestamp_ny'] = df_dxy.index.tz_convert('America/New_York')
    
    # Filtro de fin de semana
    ny_dt = df_dxy['timestamp_ny']
    is_weekend = (
        ((ny_dt.dt.weekday == 4) & (ny_dt.dt.hour >= 17)) | 
        (ny_dt.dt.weekday == 5) |                         
        ((ny_dt.dt.weekday == 6) & (ny_dt.dt.hour < 17))    
    )
    df_dxy = df_dxy[~is_weekend]
    
    output_path = os.path.join(PROCESSED_INDICES_PATH, "dxy_index.parquet")
    df_dxy.to_parquet(output_path, compression='snappy')
    print(f"✅ DXY Index generado y guardado en: {output_path}")
    
    return df_dxy

def run_correlation_engine():
    """
    Motor de Análisis de Correlación Dinámica XAUUSD vs DXY.
    Carga absoluta desde archivos Parquet.
    """
    print(f"\n--- INICIANDO MOTOR DE CORRELACIÓN DINÁMICA (v3.0) ---")
    
    gold_path = os.path.join(PROCESSED_FOREX_PATH, "xauusd_clean.parquet")
    dxy_path = os.path.join(PROCESSED_INDICES_PATH, "dxy_index.parquet")
    
    if not os.path.exists(gold_path) or not os.path.exists(dxy_path):
        print(f"❌ Error: Faltan archivos Parquet críticos para el análisis.")
        return

    print("Cargando Datasets Parquet...")
    df_gold = pd.read_parquet(gold_path, columns=['Close', 'timestamp_ny'])
    df_dxy = pd.read_parquet(dxy_path, columns=['Close'])
    
    print(f"  ✅ Carga exitosa DXY: {dxy_path} ({len(df_dxy):,} registros)")
    
    # Sincronización temporal
    df = df_gold.join(df_dxy, lsuffix='_gold', rsuffix='_dxy', how='inner')
    
    if df.empty:
        print("❌ Error: No hay coincidencia temporal entre los Parquets.")
        return

    # Cálculos estadísticos
    df['ret_gold'] = np.log(df['Close_gold'] / df['Close_gold'].shift(1))
    df['ret_dxy'] = np.log(df['Close_dxy'] / df['Close_dxy'].shift(1))
    
    print(f"Calculando correlaciones sobre {len(df):,} registros sincronizados...")
    df['corr_60'] = df['ret_gold'].rolling(window=60, min_periods=50).corr(df['ret_dxy'])
    
    # Detección de anomalías (Ruptura de correlación inversa típica)
    df['is_anomaly'] = df['corr_60'] > -0.30 
    
    # Guardado de análisis final
    analysis_path = os.path.join(PROCESSED_INDICES_PATH, "gold_dxy_analysis.parquet")
    df.to_parquet(analysis_path, compression='snappy')
    
    pct_anomaly = (df['is_anomaly'].sum() / len(df)) * 100
    print(f"✅ Análisis completado.")
    print(f"📊 Tiempo en Anomalía: {pct_anomaly:.2f}%")
    print(f"📂 Archivo generado: {analysis_path}")

def run_pipeline():
    """
    Pipeline principal: Solo Parquet.
    """
    # 1. Cargar Oro (desde Parquet)
    load_gold_asset("xauusd")
    
    # 2. Verificar/Calcular DXY (desde Parquets de componentes)
    if not os.path.exists(os.path.join(PROCESSED_INDICES_PATH, "dxy_index.parquet")):
        calculate_dxy_from_parquet()
    
    # 3. Correlación
    run_correlation_engine()

if __name__ == "__main__":
    run_pipeline()

--- INICIANDO CAPA GOLD (CARGA DE ACTIVOS) ---
📦 Fuente: Parquet. Cargando XAUUSD...
  ✅ Cargado: /home/quant/data/processed/forex/xauusd_clean.parquet (7,730,082 registros)

--- INICIANDO MOTOR DE CORRELACIÓN DINÁMICA (v3.0) ---
Cargando Datasets Parquet...
  ✅ Carga exitosa DXY: /home/quant/data/processed/indices/dxy_index.parquet (6,687,801 registros)
Calculando correlaciones sobre 6,131,578 registros sincronizados...
✅ Análisis completado.
📊 Tiempo en Anomalía: 53.11%
📂 Archivo generado: /home/quant/data/processed/indices/gold_dxy_analysis.parquet
